In [ ]:
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from statsmodels.nonparametric.smoothers_lowess import lowess
# All data paths are relative to Downloads/mouse
os.chdir("/Users/apple/Downloads/mouse")
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
sc.settings.verbosity = 1
COLOR_MD = "#4C78A8"      # ModuleDrops
COLOR_KNEE = "#F58518"    # Knee / Hard cutoff
COLOR_SHARED = "#B0B0B0"  # shared by both methods
# variable names used in the original notebooks
color_md, color_knee, color_shared = COLOR_MD, COLOR_KNEE, COLOR_SHARED
# species UMAP results (MD = ModuleDrops, HC = Hard cutoff)
wt_md_umap = sc.read_h5ad("wt.md_umap.h5ad")
wt_hc_umap = sc.read_h5ad("wt.hc_umap.h5ad")
db_md_umap = sc.read_h5ad("db.md_umap.h5ad")
db_hc_umap = sc.read_h5ad("db.hc_umap.h5ad")
# one shared species palette so panels are comparable
_species = sorted(set(wt_md_umap.obs["species"]).union(db_md_umap.obs["species"]))
_TAB = plt.get_cmap("tab20").colors + plt.get_cmap("tab20b").colors
species_colors_unified = {sp: _TAB[k % len(_TAB)] for k, sp in enumerate(_species)}
species_colors_unified.setdefault("Other", "#D9D9D9")
print(f"Number of species: {len(_species)}")


In [ ]:
# ===== panel a (left/right): WT species composition UMAP (ModuleDrops / Hard cutoff) =====
adata_wt = wt_md_umap.copy()
n_species = adata_wt.obs["species"].nunique()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
fig, ax = plt.subplots(figsize=(11, 5))
# keep species with >=1% abundance, others are merged into "Other"
species_counts = adata_wt.obs["species"].value_counts()
rare_species = species_counts[species_counts / species_counts.sum() < 0.01].index
adata_wt.obs["species_plot"] = adata_wt.obs["species"].astype(str)
adata_wt = adata_wt[~adata_wt.obs["species"].isin(rare_species)].copy()
sc.pl.umap(
    adata_wt,
    color="species_plot",
    palette=species_colors_unified,
    legend_loc="right margin",
    size=15,
    frameon=True,
    title="Species UMAP - WT MD",
    ax=ax,
    show=False,
)
for collection in ax.collections:
    collection.set_rasterized(True)
print(f"Number of species: {n_species}")
plt.tight_layout()
plt.show()
#note4fig: fig6a_part2


In [ ]:
adata_wt = wt_hc_umap.copy()
n_species = adata_wt.obs["species"].nunique()
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
fig, ax = plt.subplots(figsize=(11, 5))
# keep species with >=1% abundance, others are merged into "Other"
species_counts = adata_wt.obs["species"].value_counts()
rare_species = species_counts[species_counts / species_counts.sum() < 0.01].index
adata_wt.obs["species_plot"] = adata_wt.obs["species"].astype(str)
adata_wt = adata_wt[~adata_wt.obs["species"].isin(rare_species)].copy()
sc.pl.umap(
    adata_wt,
    color="species_plot",
    palette=species_colors_unified,
    legend_loc="right margin",
    size=15,
    frameon=True,
    title="Species UMAP - WT Knee",
    ax=ax,
    show=False
)
for collection in ax.collections:
    collection.set_rasterized(True)
print(f"Number of species: {n_species}")
plt.tight_layout()
plt.show()
#note4fig: fig6a_part1


In [ ]:
# ===== panel b (left/right): DB species composition UMAP (ModuleDrops / Hard cutoff) =====
adata_db = db_md_umap.copy()
n_species = adata_db.obs["species"].nunique()
fig, ax = plt.subplots(figsize=(8, 5))
species_counts = adata_db.obs["species"].value_counts()
rare_species = species_counts[species_counts / species_counts.sum() < 0.01].index
adata_db.obs["species_plot"] = adata_db.obs["species"].astype(str)
adata_db = adata_db[~adata_db.obs["species"].isin(rare_species)].copy()
sc.pl.umap(
    adata_db,
    color="species_plot",
    palette=species_colors_unified,
    legend_loc="right margin",
    size=15,
    frameon=True,
    title="Species UMAP - DB MD",
    ax=ax,
    show=False
)
legend = ax.get_legend()
if legend is not None:
    legend.set_ncols(2)
    # keep legend font size fixed
    for text in legend.get_texts():
        text.set_fontsize(8)
for collection in ax.collections:
    collection.set_rasterized(True)
print(f"Number of species: {n_species}")
plt.tight_layout()
plt.show()
#note4fig: fig6b_part2


In [ ]:
# copy the DB Hard cutoff species UMAP result
adata_db = db_hc_umap.copy()
adata_db = sc.read_h5ad('db.hc_umap.h5ad')
n_species = adata_db.obs["species"].nunique()
fig, ax = plt.subplots(figsize=(11, 5))
species_counts = adata_db.obs["species"].value_counts()
rare_species = species_counts[species_counts / species_counts.sum() < 0.01].index
adata_db.obs["species_plot"] = adata_db.obs["species"].astype(str)
adata_db = adata_db[~adata_db.obs["species"].isin(rare_species)].copy()
sc.pl.umap(
    adata_db,
    color="species_plot",
    palette=species_colors_unified,
    legend_loc="right margin",
    size=15,
    frameon=True,
    title="Species UMAP - DB Knee",
    ax=ax,
    show=False
)
for collection in ax.collections:
    collection.set_rasterized(True)
print(f"Number of species: {n_species}")
plt.tight_layout()
plt.show()
#note4fig: fig6b_part1


In [ ]:
# ===== panel c/d left: barcode-rank curves for WT and DB (LOWESS + ModuleDrops + Knee) =====
wt_md = pd.read_csv('wt.md.tsv', sep='\t')
wt_knee = pd.read_csv('wt.knee.tsv', sep='\t')
db_md = pd.read_csv('db.md.tsv', sep='\t')
db_knee = pd.read_csv('db.knee.tsv', sep='\t')
wt_md_cells = set(wt_md['cell'].unique())
wt_knee_cells = set(wt_knee['cell'].unique())
db_md_cells = set(db_md['cell'].unique())
db_knee_cells = set(db_knee['cell'].unique())
print(f'WT ModuleDrops: {len(wt_md_cells):,} cells')
print(f'WT Knee:        {len(wt_knee_cells):,} cells')
print(f'DB ModuleDrops: {len(db_md_cells):,} cells')
print(f'DB Knee:        {len(db_knee_cells):,} cells')


In [ ]:
# read WT counts table
df_wt = pd.read_csv('counts_wt.tsv', sep='\t')
print(f'WT total barcodes: {len(df_wt):,}')
print(df_wt.head())
df_wt_sorted = df_wt.sort_values('umi_count', ascending=False).reset_index(drop=True)
df_wt_sorted['rank'] = np.arange(1, len(df_wt_sorted) + 1)
df_wt_sorted['log_rank'] = np.log10(df_wt_sorted['rank'])
df_wt_sorted['log_umi'] = np.log10(df_wt_sorted['umi_count'])
md_match_wt = df_wt_sorted['barcode'].isin(wt_md_cells)
knee_match_wt = df_wt_sorted['barcode'].isin(wt_knee_cells)
md_plot_wt = df_wt_sorted[md_match_wt].copy()
knee_plot_wt = df_wt_sorted[knee_match_wt].copy()
print(f'WT ModuleDrops matched: {len(md_plot_wt):,}')
print(f'WT Knee matched:         {len(knee_plot_wt):,}')


In [ ]:
# WT: barcode-rank curve and knee point
step = 100
sample_idx = np.arange(0, len(df_wt_sorted), step)
log_rank_sample = df_wt_sorted.loc[sample_idx, 'log_rank'].values
log_umi_sample = df_wt_sorted.loc[sample_idx, 'log_umi'].values
lowess_fit_wt = lowess(endog=log_umi_sample,exog=log_rank_sample,frac=0.02,it=3,return_sorted=True,)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(
    lowess_fit_wt[:, 0], lowess_fit_wt[:, 1],
    color='black', linewidth=1.5, label='LOWESS', zorder=1
)
ax.scatter(
    md_plot_wt['log_rank'], md_plot_wt['log_umi'],
    s=14, color=color_md, alpha=0.8, edgecolors='none',
    label='ModuleDrops', zorder=3, rasterized=True
)
ax.scatter(
    knee_plot_wt['log_rank'], knee_plot_wt['log_umi'],
    s=14, color=color_knee, alpha=0.8, edgecolors='none',
    label='Knee', zorder=4, rasterized=True
)
ax.set_xlabel('log10(Barcode Rank)', fontsize=12)
ax.set_ylabel('log10(Total UMI Count)', fontsize=12)
ax.set_title('WT Barcode Distribution', fontsize=14)
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.show()
#note4fig: fig6c_part1


In [ ]:
# read DB counts table
df_db = pd.read_csv('counts_db.tsv', sep='\t')
print(f'DB total barcodes: {len(df_db):,}')
print(df_db.head())
df_db_sorted = df_db.sort_values('umi_count', ascending=False).reset_index(drop=True)
df_db_sorted['rank'] = np.arange(1, len(df_db_sorted) + 1)
df_db_sorted['log_rank'] = np.log10(df_db_sorted['rank'])
df_db_sorted['log_umi'] = np.log10(df_db_sorted['umi_count'])
md_match_db = df_db_sorted['barcode'].isin(db_md_cells)
knee_match_db = df_db_sorted['barcode'].isin(db_knee_cells)
md_plot_db = df_db_sorted[md_match_db].copy()
knee_plot_db = df_db_sorted[knee_match_db].copy()
print(f'DB ModuleDrops matched: {len(md_plot_db):,}')
print(f'DB Knee matched:         {len(knee_plot_db):,}')


In [ ]:
# DB: barcode-rank curve and knee point
sample_idx_db = np.arange(0, len(df_db_sorted), step)
log_rank_sample_db = df_db_sorted.loc[sample_idx_db, 'log_rank'].values
log_umi_sample_db = df_db_sorted.loc[sample_idx_db, 'log_umi'].values
lowess_fit_db = lowess(
    endog=log_umi_sample_db,
    exog=log_rank_sample_db,
    frac=0.02,
    it=3,
    return_sorted=True
)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(
    lowess_fit_db[:, 0], lowess_fit_db[:, 1],
    color='black', linewidth=1.5, label='LOWESS', zorder=1
)
ax.scatter(
    md_plot_db['log_rank'], md_plot_db['log_umi'],
    s=14, color=color_md, alpha=0.8, edgecolors='none',
    label='ModuleDrops', zorder=3, rasterized=True
)
ax.scatter(
    knee_plot_db['log_rank'], knee_plot_db['log_umi'],
    s=14, color=color_knee, alpha=0.8, edgecolors='none',
    label='Knee', zorder=4, rasterized=True
)
ax.set_xlabel('log10(Barcode Rank)', fontsize=12)
ax.set_ylabel('log10(Total UMI Count)', fontsize=12)
ax.set_title('DB Barcode Distribution', fontsize=14)
ax.legend(frameon=False, fontsize=10)
plt.tight_layout()
plt.show()
#note4fig: fig6d_part1


In [ ]:
# ===== panel c/d right: cell calling Venn diagrams for WT and DB =====
fig, axes = plt.subplots(1, 2, figsize=(9, 4), dpi=300)
for ax, md_cells, knee_cells, title in [
    (axes[0], wt_md_cells, wt_knee_cells, 'WT'),
    (axes[1], db_md_cells, db_knee_cells, 'DB'),
]:
    v = venn2(
        [md_cells, knee_cells],
        set_labels=('ModuleDrops', 'Knee'),
        ax=ax
    )
    if v.get_patch_by_id('10') is not None:
        v.get_patch_by_id('10').set_color(color_md)
        v.get_patch_by_id('10').set_alpha(0.55)
    if v.get_patch_by_id('01') is not None:
        v.get_patch_by_id('01').set_color(color_knee)
        v.get_patch_by_id('01').set_alpha(0.55)
    if v.get_patch_by_id('11') is not None:
        v.get_patch_by_id('11').set_color(color_shared)
        v.get_patch_by_id('11').set_alpha(0.65)
    for text in v.set_labels:
        if text is not None:
            text.set_fontsize(10)
    for text in v.subset_labels:
        if text is not None:
            text.set_fontsize(10)
    ax.set_title(f'{title} — cell calling overlap', fontsize=11)
plt.tight_layout()
plt.show()
#note4fig: fig6c_part2 and fig6d_part2


In [ ]:
# ===== panel e/f: calling status UMAP (shared / ModuleDrops-only / Hard cutoff-only) =====
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Load pre-computed h5ad (with UMAP already calculated)
# From mouse.ipynb (MD = ModuleDrops)
wt_md = sc.read_h5ad('wt.md_umap.h5ad')
db_md = sc.read_h5ad('db.md_umap.h5ad')
# From mouse_hardcutoff.ipynb (HC = Hard Cutoff / knee)
wt_hc = sc.read_h5ad('wt.hc_umap.h5ad')
db_hc = sc.read_h5ad('db.hc_umap.h5ad')
print(f'WT MD: {wt_md.shape}')
print(f'DB MD: {db_md.shape}')
print(f'WT HC: {wt_hc.shape}')
print(f'DB HC: {db_hc.shape}')
# Verify UMAP is present
print(f"\nWT MD has X_umap: {'X_umap' in wt_md.obsm}")
print(f"WT MD has MAG_fraction: {'MAG_fraction' in wt_md.obs}")


In [ ]:
# define calling status annotation and shared-neighbour helpers
def strip_prefix(names):
    """Strip batch prefix to get original barcode for comparison."""
    return pd.Index([n.split('_', 1)[-1] if '_' in n else n for n in names])
def annotate_calling_status(adata_md, adata_hc):
    """Annotate MD ref and HC ref with calling_status.
    Returns (ref_md, ref_hc, shared, md_only, hc_only).
    Categories are ordered: shared (first, bottom layer), method_only (last, top layer)."""
    bc_md = set(strip_prefix(adata_md.obs_names))
    bc_hc = set(strip_prefix(adata_hc.obs_names))
    shared = bc_md & bc_hc
    md_only = bc_md - bc_hc
    hc_only = bc_hc - bc_md
    # MD reference
    ref_md = adata_md.copy()
    orig_bc_md = strip_prefix(ref_md.obs_names)
    ref_md.obs['calling_status'] = 'shared'
    ref_md.obs.loc[np.isin(orig_bc_md, list(md_only)), 'calling_status'] = 'MD_only'
    ref_md.obs['calling_status'] = pd.Categorical(
        ref_md.obs['calling_status'],
        categories=['shared', 'MD_only'],
        ordered=True
    )
    # HC reference
    ref_hc = adata_hc.copy()
    orig_bc_hc = strip_prefix(ref_hc.obs_names)
    ref_hc.obs['calling_status'] = 'shared'
    ref_hc.obs.loc[np.isin(orig_bc_hc, list(hc_only)), 'calling_status'] = 'HC_only'
    ref_hc.obs['calling_status'] = pd.Categorical(
        ref_hc.obs['calling_status'],
        categories=['shared', 'HC_only'],
        ordered=True
    )
    return ref_md, ref_hc, shared, md_only, hc_only
def plot_status_umap(ref_md, ref_hc, title_prefix, ax_left, ax_right):
    """Plot MD and HC reference UMAPs side by side with calling_status colors."""
    # MD: shared=gray, MD_only=blue
    colors_md = {'shared': '#B0B0B0', 'MD_only': '#4C78A8'}
    ref_md.uns['calling_status_colors'] = [
        colors_md[x] for x in ref_md.obs['calling_status'].cat.categories
    ]
    sc.pl.umap(ref_md, color='calling_status', ax=ax_left, show=False,
               size=10, frameon=False, title=f'{title_prefix} MD reference')
    # HC: shared=gray, HC_only=red
    colors_hc = {'shared': '#B0B0B0', 'HC_only': '#F58518'}
    ref_hc.uns['calling_status_colors'] = [
        colors_hc[x] for x in ref_hc.obs['calling_status'].cat.categories
    ]
    sc.pl.umap(ref_hc, color='calling_status', ax=ax_right, show=False,
               size=10, frameon=False, title=f'{title_prefix} HC reference')


In [ ]:
# WT: annotate MD/HC calling status
wt_ref_md, wt_ref_hc, wt_shared, wt_md_only, wt_hc_only = annotate_calling_status(wt_md, wt_hc)
print(f'WT MD cells: {len(wt_ref_md):,}')
print(f'WT HC cells: {len(wt_ref_hc):,}')
print(f'WT shared: {len(wt_shared):,}')
print(f'WT MD-only: {len(wt_md_only):,}')
print(f'WT HC-only: {len(wt_hc_only):,}')


In [ ]:
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_status_umap(wt_ref_md, wt_ref_hc, 'WT', axes[0], axes[1])
for ax in axes:
    # show frame
    ax.set_frame_on(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
for ax in axes:
    for collection in ax.collections:
        collection.set_rasterized(True)
plt.tight_layout()
plt.show()
#note4fig: fig6e


In [ ]:
# DB: annotate MD/HC calling status
db_ref_md, db_ref_hc, db_shared, db_md_only, db_hc_only = annotate_calling_status(db_md, db_hc)
print(f'DB MD cells: {len(db_ref_md):,}')
print(f'DB HC cells: {len(db_ref_hc):,}')
print(f'DB shared: {len(db_shared):,}')
print(f'DB MD-only: {len(db_md_only):,}')
print(f'DB HC-only: {len(db_hc_only):,}')


In [ ]:
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Helvetica", "DejaVu Sans"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_status_umap(db_ref_md, db_ref_hc, 'DB', axes[0], axes[1])
for ax in axes:
    # show frame
    ax.set_frame_on(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
for ax in axes:
    for collection in ax.collections:
        collection.set_rasterized(True)
plt.tight_layout()
plt.show()
#note4fig: fig6f


In [ ]:
# ===== panel g/h: shared-neighbour fraction of method-only barcodes =====
from sklearn.neighbors import NearestNeighbors
def compute_neighbor_consistency(adata_ref, only_label):
    """Count shared neighbors out of 50 nearest for method-only cells."""
    X_pca = adata_ref.obsm['X_pca'][:, :30]
    nbrs = NearestNeighbors(n_neighbors=51, metric='euclidean')
    nbrs.fit(X_pca)
    distances, indices = nbrs.kneighbors(X_pca)
    only_mask = adata_ref.obs['calling_status'] == only_label
    only_indices = np.where(only_mask)[0]
    counts = []
    for idx in only_indices:
        nbr_idx = indices[idx][1:]  # skip self
        statuses = adata_ref.obs['calling_status'].iloc[nbr_idx]
        counts.append((statuses == 'shared').sum())
    return np.array(counts)
wt_md_nbrs = compute_neighbor_consistency(wt_ref_md, 'MD_only')
wt_hc_nbrs = compute_neighbor_consistency(wt_ref_hc, 'HC_only')
db_md_nbrs = compute_neighbor_consistency(db_ref_md, 'MD_only')
db_hc_nbrs = compute_neighbor_consistency(db_ref_hc, 'HC_only')
print(f'WT MD-only: mean={wt_md_nbrs.mean():.1f}/50 shared neighbors')
print(f'WT HC-only: mean={wt_hc_nbrs.mean():.1f}/50 shared neighbors')
print(f'DB MD-only: mean={db_md_nbrs.mean():.1f}/50 shared neighbors')
print(f'DB HC-only: mean={db_hc_nbrs.mean():.1f}/50 shared neighbors')


In [ ]:
# plot shared-neighbour distributions for WT/DB method-only barcodes
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
datasets = [
    (axes[0,0], wt_md_nbrs, '#4C78A8', 'WT MD-only'),
    (axes[0,1], wt_hc_nbrs, '#F58518', 'WT HC-only'),
    (axes[1,0], db_md_nbrs, '#4C78A8', 'DB MD-only'),
    (axes[1,1], db_hc_nbrs, '#F58518', 'DB HC-only'),
]
for ax, data, color, title in datasets:
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='black', linestyle='--', label=f'Mean={data.mean():.1f}')
    ax.set_xlabel('Shared neighbors (out of 50)')
    ax.set_ylabel('Cells')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ===== panel i: purity distribution of method-only barcodes (violin) =====
os.chdir("mgor")
gene_biotype = {}
gene_name_map = {}  # gene_id -> gene_name (from Name or gene attribute)
with open('mgor.gtf') as f:
    for line in f:
        if line.startswith('#'):
            continue
        cols = line.rstrip().split('\t')
        if len(cols) < 9:
            continue
        m_id = re.search(r'gene_id "([^"]+)"', cols[8])
        m_bt = re.search(r'gene_biotype "([^"]+)"', cols[8])
        m_name = re.search(r'Name=([^;]+)', cols[8])
        if not m_name:
            m_name = re.search(r'gene "([^"]+)"', cols[8])
        if m_id and m_bt:
            gene_biotype[m_id.group(1)] = m_bt.group(1)
            if m_name:
                gene_name_map[m_id.group(1)] = m_name.group(1)
print(f'Total genes in GTF: {len(gene_biotype)}')
print('Biotype distribution:')
for bt, cnt in Counter(gene_biotype.values()).most_common():
    print(f'  {bt}: {cnt}')
print(f'\nGenes with name annotation: {len(gene_name_map)}')


In [ ]:
def load_tsv(path):
    """Load tri-format tsv (gene, cell, count) into AnnData."""
    df = pd.read_csv(path, sep='\t')
    mat = df.pivot_table(index='cell', columns='gene', values='count', fill_value=0)
    adata = sc.AnnData(mat.astype('float32'))
    adata.layers['counts'] = adata.X.copy()
    adata.obs['total_umis'] = adata.X.sum(axis=1).A1 if hasattr(adata.X, 'A1') else adata.X.sum(axis=1)
    return adata
def filter_coding_genes(adata, gene_biotype):
    """Keep only protein_coding genes."""
    keep = [g for g in adata.var_names if gene_biotype.get(g) == 'protein_coding']
    adata = adata[:, keep].copy()
    return adata
def add_gene_names(adata, gene_name_map):
    adata.var['gene_name'] = adata.var_names.map(gene_name_map)
    # fall back to gene_id when no gene name is available
    missing = adata.var['gene_name'].isna()
    adata.var.loc[missing, 'gene_name'] = adata.var_names[missing]
    return adata
def preprocess(adata, n_hvg=3000, n_pcs=30):
    """Normalize, HVG, PCA, neighbors, UMAP, Leiden."""
    sc.pp.filter_cells(adata, min_genes=3)
    sc.pp.filter_genes(adata, min_cells=1)
    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    n_hvg = min(n_hvg, adata.shape[1])
    sc.pp.highly_variable_genes(adata, n_top_genes=n_hvg, flavor='cell_ranger')
    n_pcs = min(n_pcs, adata.shape[1] - 1)
    sc.tl.pca(adata, n_comps=n_pcs, use_highly_variable=True)
    sc.pp.neighbors(adata, n_neighbors=50, n_pcs=n_pcs)
    sc.tl.umap(adata, random_state=0,min_dist=1)
    sc.tl.leiden(adata, resolution=0.3, random_state=0)
    return adata
def annotate_calling_status(adata_md, adata_hc):
    """Annotate shared / method-only cells for both methods.
    Returns (adata_md_annotated, adata_hc_annotated, shared_set, md_only_set, hc_only_set)
    """
    md_bc = set(adata_md.obs_names)
    hc_bc = set(adata_hc.obs_names)
    shared = md_bc & hc_bc
    md_only = md_bc - hc_bc
    hc_only = hc_bc - md_bc
    # MD reference
    adata_md = adata_md.copy()
    adata_md.obs['calling_status'] = 'shared'
    adata_md.obs.loc[list(md_only), 'calling_status'] = 'MD_only'
    adata_md.obs['calling_status'] = pd.Categorical(
        adata_md.obs['calling_status'],
        categories=['shared', 'MD_only'],
        ordered=True
    )
    # HC reference
    adata_hc = adata_hc.copy()
    adata_hc.obs['calling_status'] = 'shared'
    adata_hc.obs.loc[list(hc_only), 'calling_status'] = 'HC_only'
    adata_hc.obs['calling_status'] = pd.Categorical(
        adata_hc.obs['calling_status'],
        categories=['shared', 'HC_only'],
        ordered=True
    )
    return adata_md, adata_hc, shared, md_only, hc_only
def plot_status_umap(adata_ref_md, adata_ref_hc, sample_name, axes):
    """Plot calling status UMAP for both references side by side."""
    colors_md = {'shared': 'lightgrey', 'MD_only': '#4C78A8'}
    colors_hc = {'shared': 'lightgrey', 'HC_only': '#E45756'}
    # MD reference
    adata_ref_md.uns['calling_status_colors'] = [
        colors_md[x] for x in adata_ref_md.obs['calling_status'].cat.categories
    ]
    sc.pl.umap(
        adata_ref_md,
        color='calling_status',
        ax=axes[0],
        show=False,
        size=30,
        frameon=True,
        title=f'{sample_name} — MD reference'
    )
    # HC reference
    adata_ref_hc.uns['calling_status_colors'] = [
        colors_hc[x] for x in adata_ref_hc.obs['calling_status'].cat.categories
    ]
    sc.pl.umap(
        adata_ref_hc,
        color='calling_status',
        ax=axes[1],
        show=False,
        size=30,
        frameon=True,
        title=f'{sample_name} — HC reference'
    )
    # Rasterize
    for ax in axes:
        for collection in ax.collections:
            collection.set_rasterized(True)


In [ ]:
# load WT MD/HC matrices, keep protein-coding genes only
# Load data
wt_md_raw = load_tsv('wt.md.tsv')
wt_hc_raw = load_tsv('wt.hc.tsv')
print(f'WT MD raw: {wt_md_raw.shape}')
print(f'WT HC raw: {wt_hc_raw.shape}')
# Keep only protein coding
wt_md = filter_coding_genes(wt_md_raw, gene_biotype)
wt_hc = filter_coding_genes(wt_hc_raw, gene_biotype)
print(f'WT MD coding: {wt_md.shape}')
print(f'WT HC coding: {wt_hc.shape}')
# Add gene names
wt_md = add_gene_names(wt_md, gene_name_map)
wt_hc = add_gene_names(wt_hc, gene_name_map)


In [ ]:
# WT: preprocess each method separately (HVG + PCA + neighbour graph)
# Preprocess both independently
wt_md = preprocess(wt_md, n_hvg=2000, n_pcs=30)
wt_hc = preprocess(wt_hc, n_hvg=2000, n_pcs=30)
print(f'WT MD after preprocessing: {wt_md.shape}')
print(f'WT HC after preprocessing: {wt_hc.shape}')
print(f'WT MD Leiden clusters: {wt_md.obs["leiden"].nunique()}')
print(f'WT HC Leiden clusters: {wt_hc.obs["leiden"].nunique()}')


In [ ]:
# WT: annotate calling status
# Annotate calling status
wt_ref_md, wt_ref_hc, wt_shared, wt_md_only, wt_hc_only = annotate_calling_status(wt_md, wt_hc)
print('WT MD reference:')
print(wt_ref_md.obs['calling_status'].value_counts())
print()
print('WT HC reference:')
print(wt_ref_hc.obs['calling_status'].value_counts())


In [ ]:
# load DB MD/HC matrices, keep protein-coding genes only
# Load data
db_md_raw = load_tsv('db.md.tsv')
db_hc_raw = load_tsv('db.hc.tsv')
print(f'DB MD raw: {db_md_raw.shape}')
print(f'DB HC raw: {db_hc_raw.shape}')
# Keep only protein coding
db_md = filter_coding_genes(db_md_raw, gene_biotype)
db_hc = filter_coding_genes(db_hc_raw, gene_biotype)
print(f'DB MD coding: {db_md.shape}')
print(f'DB HC coding: {db_hc.shape}')
# Add gene names
db_md = add_gene_names(db_md, gene_name_map)
db_hc = add_gene_names(db_hc, gene_name_map)


In [ ]:
# DB: preprocess each method separately (HVG + PCA + neighbour graph)
# Preprocess both independently
db_md = preprocess(db_md, n_hvg=2000, n_pcs=30)
db_hc = preprocess(db_hc, n_hvg=2000, n_pcs=30)
print(f'DB MD after preprocessing: {db_md.shape}')
print(f'DB HC after preprocessing: {db_hc.shape}')
print(f'DB MD Leiden clusters: {db_md.obs["leiden"].nunique()}')
print(f'DB HC Leiden clusters: {db_hc.obs["leiden"].nunique()}')


In [ ]:
# DB: annotate calling status
# Annotate calling status
db_ref_md, db_ref_hc, db_shared, db_md_only, db_hc_only = annotate_calling_status(db_md, db_hc)
print('DB MD reference:')
print(db_ref_md.obs['calling_status'].value_counts())
print()
print('DB HC reference:')
print(db_ref_hc.obs['calling_status'].value_counts())


In [ ]:
# compute per-barcode purity from BAM alignments
# ============================================================
# Purity comparison: method-only barcodes (from BAM)
# purity = mapped reads / total reads per barcode
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
# ---- Load BAM-based purity per barcode ----
wt_purity = pd.read_csv("wt.purity.tsv", sep="\t")
db_purity = pd.read_csv("db.purity.tsv", sep="\t")
wt_pur_map = wt_purity.set_index("bc")["purity"]
db_pur_map = db_purity.set_index("bc")["purity"]
print(f"WT barcodes with purity: {len(wt_pur_map)}")
print(f"DB barcodes with purity: {len(db_pur_map)}")
# ---- Look up purity for method-only barcodes ----
wt_md_only_purity = wt_pur_map.reindex(list(wt_md_only)).dropna().values
wt_hc_only_purity = wt_pur_map.reindex(list(wt_hc_only)).dropna().values
db_md_only_purity = db_pur_map.reindex(list(db_md_only)).dropna().values
db_hc_only_purity = db_pur_map.reindex(list(db_hc_only)).dropna().values
print(f"\nWT MD-only: {len(wt_md_only)} barcodes, {len(wt_md_only_purity)} with purity, median={np.median(wt_md_only_purity):.4f}")
print(f"WT HC-only: {len(wt_hc_only)} barcodes, {len(wt_hc_only_purity)} with purity, median={np.median(wt_hc_only_purity):.4f}")
print(f"DB MD-only: {len(db_md_only)} barcodes, {len(db_md_only_purity)} with purity, median={np.median(db_md_only_purity):.4f}")
print(f"DB HC-only: {len(db_hc_only)} barcodes, {len(db_hc_only_purity)} with purity, median={np.median(db_hc_only_purity):.4f}")
# ---- Violin plot ----
groups = [
    ("WT", "HC-only", wt_hc_only_purity),
    ("WT", "MD-only", wt_md_only_purity),
    ("DB", "HC-only", db_hc_only_purity),
    ("DB", "MD-only", db_md_only_purity),
]
plot_data = pd.DataFrame(
    [{"sample": s, "method": m, "purity": v} for s, m, arr in groups for v in arr]
)
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
colors = {"MD-only": "#4C78A8", "HC-only": "#E45756"}
sns.violinplot(
    data=plot_data,
    x="sample",
    y="purity",
    hue="method",
    palette=colors,
    inner="box",
    cut=0,
    ax=ax
)
ax.set(
    ylabel="Purity (mapped reads / total reads)",
    xlabel="Sample",
    title="Purity of method-only barcodes"
)
ax.set_ylim(0, 1.05)
ax.legend(title="Method", frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
#note4fig: fig6g


In [ ]:
# ===== panel j/k: highly expressed gene bubbles for WT and DB (Infection vs MVAcall) =====
def gene_stats(adata):
    """Compute per-gene mean expression and fraction expressed."""
    X = adata.X
    if hasattr(X, 'toarray'):
        X = X.toarray()
    else:
        X = np.asarray(X)
    mean_expression = np.mean(X, axis=0)
    fraction_expressed = np.mean(X > 0, axis=0)
    result = pd.DataFrame({
        'gene_id': adata.var_names,
        'gene_name': adata.var['gene_name'].values,
        'mean_expression': mean_expression,
        'fraction_expressed': fraction_expressed
    })
    return result
wt_md_stats = gene_stats(wt_md)
wt_hc_stats = gene_stats(wt_hc)
print('=== WT MD top 20 expressed genes ===')
print(wt_md_stats.sort_values('mean_expression', ascending=False).head(20)[['gene_name', 'mean_expression', 'fraction_expressed']].to_string(index=False))
print()
print('=== WT HC top 20 expressed genes ===')
print(wt_hc_stats.sort_values('mean_expression', ascending=False).head(20)[['gene_name', 'mean_expression', 'fraction_expressed']].to_string(index=False))


In [ ]:
# WT: highly expressed gene bubble plot
# Gene bubble plot: MD vs HC for selected genes
# Use top expressed genes + some markers
top_genes = wt_md_stats.sort_values('mean_expression', ascending=False).head(15)['gene_name'].tolist()
print(f'Plotting {len(top_genes)} genes: ' + ', '.join(top_genes))
def gene_bubble_data(adata, genes, method):
    records = []
    for gene in genes:
        idx = np.where(adata.var['gene_name'].values == gene)[0]
        if len(idx) == 0:
            continue
        x = adata.X[:, idx[0]]
        if hasattr(x, 'toarray'):
            x = x.toarray().ravel()
        else:
            x = np.asarray(x).ravel()
        records.append({
            'gene': gene,
            'method': method,
            'fraction_expressed': np.mean(x > 0),
            'mean_expression': np.mean(x)
        })
    return pd.DataFrame(records)
wt_bubble = pd.concat([
    gene_bubble_data(wt_hc, top_genes, 'HardCutoff'),
    gene_bubble_data(wt_md, top_genes, 'ModuleDrops')
], ignore_index=True)
vmin = 0.2
vmax = 3.5
methods = ['HardCutoff', 'ModuleDrops']
fig, ax = plt.subplots(figsize=(5, 6))
for xi, method in enumerate(methods):
    tmp = wt_bubble[wt_bubble['method'] == method]
    y = [top_genes.index(g) for g in tmp['gene']]
    scatter = ax.scatter(
        [xi] * len(tmp), y,
        s=tmp['fraction_expressed'] * 500,
        c=tmp['mean_expression'],
        cmap='viridis', vmin=vmin, vmax=vmax,
        alpha=0.85, edgecolors='black'
    )
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods)
ax.set_yticks(range(len(top_genes)))
ax.set_yticklabels(top_genes)
ax.set_xlabel('Calling method')
ax.set_ylabel('Gene')
ax.set_xlim(-0.5, 1.5)
cbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.6)
cbar.set_label('Mean log1p normalized expression')
size_values = [0, 25, 50, 100]
size_handles = [
    ax.scatter([], [], s=pct / 100 * 500, color='gray', alpha=0.7, edgecolors='none', label=f'{pct}%')
    for pct in size_values
]
ax.legend(handles=size_handles, title='Fraction expressed',
          loc='lower left', bbox_to_anchor=(1.02, 1.0), frameon=False, scatterpoints=1)
ax.set_title('WT — top expressed genes')
plt.tight_layout()
plt.show()
#note4fig: fig6h


In [ ]:
# DB: highly expressed gene statistics
db_md_stats = gene_stats(db_md)
db_hc_stats = gene_stats(db_hc)
print('=== DB MD top 20 expressed genes ===')
print(db_md_stats.sort_values('mean_expression', ascending=False).head(20)[['gene_name', 'mean_expression', 'fraction_expressed']].to_string(index=False))
print()
print('=== DB HC top 20 expressed genes ===')
print(db_hc_stats.sort_values('mean_expression', ascending=False).head(20)[['gene_name', 'mean_expression', 'fraction_expressed']].to_string(index=False))


In [ ]:
# DB: highly expressed gene bubble plot
# Gene bubble plot: MD vs HC for DB
db_top_genes = db_md_stats.sort_values('mean_expression', ascending=False).head(15)['gene_name'].tolist()
print(f'Plotting {len(db_top_genes)} genes: ' + ', '.join(db_top_genes))
db_bubble = pd.concat([
    gene_bubble_data(db_md, db_top_genes, 'ModuleDrops'),
    gene_bubble_data(db_hc, db_top_genes, 'HardCutoff'),
], ignore_index=True)
vmin = 0.2
vmax = 3.5
methods = ['HardCutoff', 'ModuleDrops']
fig, ax = plt.subplots(figsize=(5, 6))
for xi, method in enumerate(methods):
    tmp = db_bubble[db_bubble['method'] == method]
    y = [db_top_genes.index(g) for g in tmp['gene']]
    scatter = ax.scatter(
        [xi] * len(tmp), y,
        s=tmp['fraction_expressed'] * 500,
        c=tmp['mean_expression'],
        cmap='viridis', vmin=vmin, vmax=vmax,
        alpha=0.85, edgecolors='black'
    )
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods)
ax.set_yticks(range(len(db_top_genes)))
ax.set_yticklabels(db_top_genes)
ax.set_xlabel('Calling method')
ax.set_ylabel('Gene')
ax.set_xlim(-0.5, 1.5)
cbar = plt.colorbar(scatter, ax=ax, pad=0.1, shrink=0.6)
cbar.set_label('Mean log1p normalized expression')
size_values = [0, 25, 50, 100]
size_handles = [
    ax.scatter([], [], s=pct / 100 * 500, color='gray', alpha=0.7, edgecolors='none', label=f'{pct}%')
    for pct in size_values
]
ax.legend(handles=size_handles, title='Fraction expressed',
          loc='lower left', bbox_to_anchor=(1.02, 1.0), frameon=False, scatterpoints=1)
ax.set_title('DB — top expressed genes')
plt.tight_layout()
plt.show()
#note4fig: fig6i


In [ ]:
# ===== panel l: M. gordoncarteri gene x sample x method log2FC heatmap =====
import seaborn as sns
def wt_vs_db_de(adata_wt, adata_db, method_name):
    """Compare WT vs DB for the same method, return top up/down genes."""
    # Use only genes present in both
    common_genes = adata_wt.var_names.intersection(adata_db.var_names)
    wt_stats = gene_stats(adata_wt[:, common_genes]).set_index('gene_id')
    db_stats = gene_stats(adata_db[:, common_genes]).set_index('gene_id')
    # Compute log2 fold change (WT vs DB)
    de = pd.DataFrame({
        'gene_name': wt_stats['gene_name'],
        'mean_wt': wt_stats['mean_expression'],
        'mean_db': db_stats['mean_expression'],
        'frac_wt': wt_stats['fraction_expressed'],
        'frac_db': db_stats['fraction_expressed'],
    })
    de['log2fc'] = np.log2((de['mean_wt'] + 0.01) / (de['mean_db'] + 0.01))
    # Filter: expressed in at least 10% of cells in one condition
    de = de[(de['frac_wt'] > 0.1) | (de['frac_db'] > 0.1)]
    # Top 5 up in WT and top 5 up in DB
    top_up_wt = de.sort_values('log2fc', ascending=False).head(5)
    top_up_db = de.sort_values('log2fc', ascending=True).head(5)
    return de, top_up_wt, top_up_db
# MD method: WT vs DB
md_de, md_up_wt, md_up_db = wt_vs_db_de(wt_md, db_md, 'ModuleDrops')
print('=== MD — top 5 genes higher in WT ===')
print(md_up_wt[['gene_name', 'mean_wt', 'mean_db', 'log2fc']].to_string(index=False))
print()
print('=== MD — top 5 genes higher in DB ===')
print(md_up_db[['gene_name', 'mean_wt', 'mean_db', 'log2fc']].to_string(index=False))


In [ ]:
# differential expression WT vs DB under Hard cutoff
# HC method: WT vs DB
hc_de, hc_up_wt, hc_up_db = wt_vs_db_de(wt_hc, db_hc, 'HardCutoff')
print('=== HC — top 5 genes higher in WT ===')
print(hc_up_wt[['gene_name', 'mean_wt', 'mean_db', 'log2fc']].to_string(index=False))
print()
print('=== HC — top 5 genes higher in DB ===')
print(hc_up_db[['gene_name', 'mean_wt', 'mean_db', 'log2fc']].to_string(index=False))


In [ ]:
# side-by-side MD/HC log2FC heatmap
# Combined heatmap: MD and HC side-by-side, shared genes, shared colorbar
# Sorted: up in WT (top) -> down in WT (bottom)
md_top_genes = pd.concat([md_up_wt, md_up_db])
hc_top_genes = pd.concat([hc_up_wt, hc_up_db])
# Use shared genes (intersection)
shared_gene_names = list(set(md_top_genes['gene_name']) & set(hc_top_genes['gene_name']))
print(f'Shared genes between MD and HC top DE: {len(shared_gene_names)}')
if len(shared_gene_names) < 5:
    shared_gene_names = list(set(md_top_genes['gene_name']) | set(hc_top_genes['gene_name']))
    print(f'Using union instead: {len(shared_gene_names)} genes')
# Get gene_id maps
md_gene_id_map = pd.Series(md_de.index, index=md_de['gene_name'])
hc_gene_id_map = pd.Series(hc_de.index, index=hc_de['gene_name'])
# Build heatmap data matrices
md_heatmap_data = pd.DataFrame(index=shared_gene_names, columns=['WT', 'DB'], dtype=float)
hc_heatmap_data = pd.DataFrame(index=shared_gene_names, columns=['WT', 'DB'], dtype=float)
log2fc_md = pd.Series(index=shared_gene_names, dtype=float)
for g in shared_gene_names:
    if g in md_gene_id_map:
        gid = md_gene_id_map[g]
        md_heatmap_data.loc[g, 'WT'] = md_de.loc[gid, 'mean_wt']
        md_heatmap_data.loc[g, 'DB'] = md_de.loc[gid, 'mean_db']
        log2fc_md[g] = md_de.loc[gid, 'log2fc']
    if g in hc_gene_id_map:
        gid = hc_gene_id_map[g]
        hc_heatmap_data.loc[g, 'WT'] = hc_de.loc[gid, 'mean_wt']
        hc_heatmap_data.loc[g, 'DB'] = hc_de.loc[gid, 'mean_db']
# Sort genes by log2fc (up in WT first, then down in WT)
sorted_genes = log2fc_md.sort_values(ascending=False).index.tolist()
md_heatmap_data = md_heatmap_data.loc[sorted_genes]
hc_heatmap_data = hc_heatmap_data.loc[sorted_genes]
# Shared colorbar range
all_vals = pd.concat([md_heatmap_data.stack(), hc_heatmap_data.stack()])
vmin_shared = all_vals.min()
vmax_shared = all_vals.max()
vcenter_shared = all_vals.median()
n_genes = len(sorted_genes)
fig, axes = plt.subplots(1, 2, figsize=(6, max(5, n_genes * 0.35 + 2)), sharey=True)
# HC heatmap
sns.heatmap(
    hc_heatmap_data,
    cmap='RdBu_r',
    vmin=vmin_shared,
    vmax=vmax_shared,
    center=vcenter_shared,
    ax=axes[0],
    yticklabels=True,
    xticklabels=True,
    cbar=False,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 8}
)
axes[0].set_title('HC — WT vs DB')
axes[0].set_ylabel('')
# MD heatmap
sns.heatmap(
    md_heatmap_data,
    cmap='RdBu_r',
    vmin=vmin_shared,
    vmax=vmax_shared,
    center=vcenter_shared,
    ax=axes[1],
    yticklabels=True,
    xticklabels=True,
    cbar=True,
    cbar_kws={'label': 'Mean log1p expression'},
    annot=True,
    fmt='.2f',
    annot_kws={'size': 8}
)
axes[1].set_title('MD — WT vs DB')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()
#note4fig: fig6j
